# ClaD Modular Experiment Framework

Cell execution order: 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11 → 12 → 13 → 14 (verify) → 15 (run)

## Cell 1: Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install wandb loguru sentence-transformers -q

In [ ]:
import wandb
wandb.login()
from huggingface_hub import login
login()

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

## Cell 2: Global Paths

In [ ]:
import os

BASE_DIR   = '/content/drive/MyDrive/Parth/CLEF_prvt/Fall 25'
DATA_DIR   = os.path.join(BASE_DIR, 'Data')
MODEL_DIR  = '/content/drive/MyDrive/Parth/CLEF_prvt/Spring 26_finale/Models/ClaD'

PATH_COLLECTION_DATA = os.path.join(DATA_DIR, 'collection_baseline.csv')
PATH_TRAIN_INFONCE   = os.path.join(DATA_DIR, 'ranked/Finetune/train_infonce.csv')
PATH_DEV_DATA        = os.path.join(DATA_DIR, 'raw/subtask4b_query_tweets_dev.tsv')
PATH_TEST_DATA       = os.path.join(DATA_DIR, 'raw/subtask4b_query_tweets_test.tsv')

## Cell 3: Imports

In [ ]:
import torch
import torch.nn.functional as F
import torch.nn as nn
from sentence_transformers import SentenceTransformer, models
from sentence_transformers.util import batch_to_device
from torch.utils.data import DataLoader, Sampler
from torch.optim.lr_scheduler import OneCycleLR
from typing import List, Optional, Callable, Any
from dataclasses import dataclass, field
import pandas as pd
import numpy as np
from tqdm import tqdm
from tqdm.auto import tqdm
from loguru import logger
import random
import os
import tempfile
from datetime import datetime
import wandb
from collections import defaultdict

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f'Device: {DEVICE}')

## Cell 4: Model Implementations

> **Must run before Cell 5** — ExperimentConfig defaults reference these functions.

Add new model types here by writing three functions: `xxx_build`, `xxx_forward`, `xxx_encode`.

In [ ]:
# ============================================================
# CELL 4: Model Implementations
# MUST be run before ExperimentConfig (Cell 5)
# ============================================================

# ------ SentenceTransformer family  ------

# def st_build(cfg, device):
#     '''Load SentenceTransformer and append ClaD projection head + normalize.'''
#     model     = SentenceTransformer(cfg.model_id)
#     emb_dim   = model.get_sentence_embedding_dimension()
#     # concat operation: pair_dim = 2 * emb_dim, so project each half to 128
#     # diff/sum operation: pair_dim = emb_dim, project to 256
#     output_dim = 128 if cfg.operation == 'concat' else 256
#     model.add_module('clad_projection', models.Dense(
#         in_features=emb_dim, out_features=output_dim,
#         bias=False, activation_function=None
#     ))
#     if cfg.projection_dropout > 0:
#         model.add_module('proj_dropout', nn.Dropout(p=cfg.projection_dropout))
#     model.add_module('clad_normalize', models.Normalize())
#     model = model.to(device)
#     logger.info(f'[st_build] {cfg.model_id}: {emb_dim}d -> {output_dim}d, '
#                 f'params={sum(p.numel() for p in model.parameters()):,}')
#     return model


# def st_forward(texts, model, device, cfg):
#     '''Single-batch forward pass with gradient. Used in training loop.'''
#     features = model.tokenize(texts, max_length=cfg.max_seq_length,
#                                truncation=True, padding='max_length')
#     features = batch_to_device(features, device)
#     return model(features)['sentence_embedding']


# def st_encode(texts, model, device, cfg, batch_size=64):
#     '''Batched encoding without gradient. Used in eval.'''
#     return F.normalize(
#         model.encode(texts, convert_to_tensor=True, batch_size=batch_size,
#                      device=device, show_progress_bar=True),
#         p=2, dim=1
#     )


# ------ HuggingFace template (raw backbones) ------
def hf_build(cfg, device):
    from transformers import AutoModel
    backbone = AutoModel.from_pretrained(cfg.model_id)
    output_dim = 128 if cfg.operation == 'concat' else 256
    backbone.clad_projection = nn.Linear(backbone.config.hidden_size, output_dim, bias=False)
    return backbone.to(device)

def hf_forward(texts, model, device, cfg):
    inputs = cfg.tokenizer(texts, return_tensors='pt', padding=True,
                            truncation=True, max_length=cfg.max_seq_length)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    out    = model(**inputs)
    emb    = out.last_hidden_state[:, 0, :]   # CLS token
    emb    = model.clad_projection(emb)
    return F.normalize(emb, p=2, dim=1)

def hf_encode(texts, model, device, cfg, batch_size=64):
    all_embs = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            all_embs.append(hf_forward(texts[i:i+batch_size], model, device, cfg))
    return torch.cat(all_embs, dim=0)

## Cell 5: ExperimentConfig

> This is the **only** thing you change between experiments.

In [ ]:
# ============================================================
# CELL 5: ExperimentConfig
# The ONLY cell you edit between experiments
# ============================================================

@dataclass
class ExperimentConfig:

    # --- Identity ---
    experiment_name: str = ''
    run_number:      int = 1
    dataset_name:    str = 'clef2025'
    wandb_group:     str = 'MiniLM'
    wandb_project:   str = 'IR_ClaD'

    # --- Model ---
    # Swap all three callables to change model type (ST vs HF vs custom)
    model_id:   str      = 'sentence-transformers/all-MiniLM-L6-v2'
    max_seq_length: int  = 512
    build_fn:   Callable = field(default=hf_build)    # (cfg, device) -> model
    forward_fn: Callable = field(default=hf_forward)  # (texts, model, device, cfg) -> Tensor[B, emb_dim]
    encode_fn:  Callable = field(default=hf_encode)   # (texts, model, device, cfg, bs) -> Tensor[N, emb_dim]
    tokenizer:  Any      = None  # only needed for raw HF models

    # --- Operation ---
    operation: str = 'concat'   # 'concat' | 'diff' | 'sum'

    # --- Training ---
    total_epochs:     int   = 10
    batch_size:       int   = 32
    grad_accum_steps: int   = 1
    seed:             int   = 42
    max_grad_norm:    float = 1.0

    # --- Loss ---
    loss_fn:     str   = 'soft_margin'  # 'soft_margin' | 'hard_margin' | 'infonce'
    margin:      float = 150.0
    temperature: float = 50.0           # for infonce (when added)

    # --- Optimizer ---
    lr:                         float = 2e-5
    scheduler_max_lr:           float = 5e-5
    scheduler_pct_start:        float = 0.3
    scheduler_div_factor:       float = 10.0
    scheduler_final_div_factor: float = 50.0
    scheduler_anneal_strategy:  str   = 'cos'

    # --- Window / Covariance ---
    window_size:             int   = 300
    regularization_term:     float = 1e-5
    use_diagonal_covariance: bool  = False

    # --- Evaluation ---
    eval_patience:          int          = 5
    eval_batch_size_query:  int          = 512
    eval_batch_size_corpus: int          = 256
    train_eval_samples:     Optional[int] = None  # None = all train queries; e.g. 1000 for speed

    # --- Regularization ---
    query_dropout:      float = 0.0    # applied to query_emb in train loop only; try 0.1–0.3
    projection_dropout: float = 0.0    # applied in st_build projection layer; try 0.1–0.2
    weight_decay:       float = 0.01   # AdamW weight_decay

    # --- Data (defaults to global paths from Cell 2) ---
    train_path:      str = ''
    dev_path:        str = ''
    collection_path: str = ''

    # --- Resume ---
    resume_model_path: Optional[str] = None
    start_epoch:       int           = 0

    # --- Hard Negatives ---
    hard_neg_path:      Optional[str] = None
    num_hard_per_query: int           = 4
    hard_neg_start:     int           = 50
    hard_neg_end:       int           = 150

    def __post_init__(self):
        if not self.experiment_name:
            model_short = self.model_id.split('/')[-1][:12]
            self.experiment_name = (
                f'exp_{self.run_number:03d}_{model_short}_{self.loss_fn}_{self.dataset_name}'
            )
        if not self.train_path:
            self.train_path = PATH_TRAIN_INFONCE
        if not self.dev_path:
            self.dev_path = PATH_DEV_DATA
        if not self.collection_path:
            self.collection_path = PATH_COLLECTION_DATA


    @property
    def save_dir(self):
        return f'{MODEL_DIR}/{self.dataset_name}/{self.experiment_name}'


    @property
    def save_path(self):
        return f'{self.save_dir}/best_model.pt'

## Cell 6: Data Utilities

In [ ]:
# ============================================================
# CELL 6: Data Utilities
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_data(train_or_not: bool, path: str, use_triplets: bool = False) -> List:
    '''
    Load data from CSV.
    Training (InfoNCE/margin): returns [(query, positive, cord_uid), ...]
    Training (triplets):       returns [(query, gold, negative), ...]
    Evaluation:                returns [(query, cord_uid), ...]
    '''
    if train_or_not:
        df = pd.read_csv(path)
        if use_triplets:
            return [(row['query'], row['gold'], row['negative']) for _, row in df.iterrows()]
        else:
            return [(row['query'], row['positive'], row['cord_uid']) for _, row in df.iterrows()]
    else:
        df = pd.read_csv(path, sep='\t')
        df['query'] = 'task: search result | query: ' + df['tweet_text']
        return [(row['query'], row['cord_uid']) for _, row in df.iterrows()]


class GreedyUniqueBatchSampler(Sampler):
    '''
    Uses ALL training pairs (zero data waste).
    Greedily builds batches ensuring unique cord_uids per batch,
    so in-batch negatives are always distinct documents.
    '''
    def __init__(self, dataset, batch_size=32, shuffle=True):
        self.batch_size = batch_size
        self.shuffle    = shuffle
        self.n          = len(dataset)
        self.idx_to_uid = {idx: str(uid) for idx, (_, _, uid) in enumerate(dataset)}

    def __iter__(self):
        remaining = set(range(self.n))
        order     = list(range(self.n))
        if self.shuffle:
            random.shuffle(order)

        while remaining:
            batch      = []
            batch_uids = set()

            for idx in order:
                if idx not in remaining:
                    continue
                uid = self.idx_to_uid[idx]
                if uid not in batch_uids and len(batch) < self.batch_size:
                    batch.append(idx)
                    batch_uids.add(uid)
                    remaining.discard(idx)
                if len(batch) == self.batch_size:
                    break

            if len(batch) >= 2:
                yield batch

            if len(remaining) < len(order) * 0.5:
                order = [i for i in order if i in remaining]

    def __len__(self):
        return (self.n + self.batch_size - 1) // self.batch_size



class HardNegBatchSampler:
    '''
    Arranges batches so hard negatives appear as in-batch negatives.
    For query q_i with hard neg d_k, places (q_k, d_k) in the same batch
    → q_i sees d_k as an in-batch negative for free (zero extra forward passes).
    '''
    def __init__(self, train_data, hard_negs, batch_size, num_hard_per_query=4):
        self.train_data = train_data
        self.batch_size = batch_size

        # Reverse map: positive_uid → list of dataset indices
        self.uid_to_indices = defaultdict(list)
        for idx, item in enumerate(train_data):
            self.uid_to_indices[str(item[2])].append(idx)

        # Sample num_hard_per_query from each query's hard negs
        self.sampled_negs = {}
        for idx, neg_uids in hard_negs.items():
            available = [uid for uid in neg_uids if uid in self.uid_to_indices]
            k = min(num_hard_per_query, len(available))
            self.sampled_negs[idx] = random.sample(available, k) if k > 0 else []

        # Adjacency: idx → set of indices whose positives are idx's hard negatives
        self.wants = defaultdict(set)
        for idx, neg_uids in self.sampled_negs.items():
            for uid in neg_uids:
                for partner in self.uid_to_indices[uid]:
                    if partner != idx:
                        self.wants[idx].add(partner)

        queries_with_negs = sum(1 for v in self.wants.values() if v)
        total_wanted = sum(len(v) for v in self.wants.values())
        logger.info(f'HardNegBatchSampler: {queries_with_negs}/{len(train_data)} queries '
                    f'have mappable hard negs, avg={total_wanted/max(queries_with_negs,1):.1f} partners')

    def __iter__(self):
        remaining = set(range(len(self.train_data)))
        batches = []

        while remaining:
            batch = []
            # Seed: query with most unfulfilled hard-neg partners still available
            seed = max(remaining, key=lambda x: len(self.wants.get(x, set()) & remaining))
            batch.append(seed); remaining.discard(seed)

            # Phase 1: add seed's hard-neg partners
            partners = list(self.wants.get(seed, set()) & remaining)
            random.shuffle(partners)
            for p in partners:
                if len(batch) >= self.batch_size: break
                batch.append(p); remaining.discard(p)

            # Phase 2: snowball — add partners' partners
            for member in list(batch[1:]):
                if len(batch) >= self.batch_size: break
                their_partners = list(self.wants.get(member, set()) & remaining)
                random.shuffle(their_partners)
                for p in their_partners:
                    if len(batch) >= self.batch_size: break
                    batch.append(p); remaining.discard(p)

            # Phase 3: fill remaining slots randomly
            fill = list(remaining); random.shuffle(fill)
            while len(batch) < self.batch_size and fill:
                idx = fill.pop(); batch.append(idx); remaining.discard(idx)

            batches.append(batch)

        random.shuffle(batches)

        # Coverage stats — logged every epoch
        total_f, total_w, per_batch = 0, 0, []
        for batch in batches:
            batch_set = set(batch)
            f = sum(len(self.wants.get(i, set()) & batch_set) for i in batch)
            w = sum(len(self.wants.get(i, set())) for i in batch)
            per_batch.append(f); total_f += f; total_w += w
        if total_w > 0:
            logger.info(f'Hard neg coverage: {total_f}/{total_w} '
                        f'({total_f/total_w*100:.1f}%) | per-batch avg={np.mean(per_batch):.1f}')

        return iter(batches)

    def __len__(self):
        return (len(self.train_data) + self.batch_size - 1) // self.batch_size

def apply_query_doc_operation(query_emb, doc_emb, operation='concat'):
    '''Combine query and doc embeddings: concat, diff, or sum.'''
    if operation == 'concat':
        return torch.cat([query_emb, doc_emb], dim=1)
    elif operation == 'diff':
        return query_emb - doc_emb
    elif operation == 'sum':
        return query_emb + doc_emb
    raise ValueError(f'Unknown operation: {operation}')

## Cell 7: Loss Functions

In [ ]:
# ============================================================
# CELL 7: Loss Functions
# ============================================================

def soft_margin_inbatch_loss(query_embs, doc_embs, mean, std,
                              covariance_matrix, margin, batch_idx, cfg):
    B        = query_embs.shape[0]
    pair_dim = mean.shape[1]

    identity = torch.eye(pair_dim, device=covariance_matrix.device) * cfg.regularization_term
    reg_cov  = covariance_matrix + identity

    if cfg.use_diagonal_covariance:
        inv_diag = 1.0 / torch.diag(reg_cov)
    else:
        inv_cov = torch.inverse(reg_cov)

    query_exp = query_embs.unsqueeze(1).expand(-1, B, -1)
    doc_exp   = doc_embs.unsqueeze(0).expand(B, -1, -1)

    if cfg.operation == 'concat':
        pairs = torch.cat([query_exp, doc_exp], dim=2)
    elif cfg.operation == 'diff':
        pairs = query_exp - doc_exp
    elif cfg.operation == 'sum':
        pairs = query_exp + doc_exp

    norm_flat = ((pairs - mean) / (std + 1e-7)).view(-1, pair_dim)

    if cfg.use_diagonal_covariance:
        mah_sq = torch.sum((norm_flat ** 2) * inv_diag, dim=1)
    else:
        mah_sq = torch.sum(norm_flat @ inv_cov * norm_flat, dim=1)

    distances     = mah_sq.view(B, B)
    pos_distances = torch.diag(distances).unsqueeze(1)
    mask          = ~torch.eye(B, dtype=torch.bool, device=distances.device)

    soft_losses = F.softplus(margin + pos_distances - distances)
    loss        = (soft_losses * mask).sum() / (B * (B - 1))

    with torch.no_grad():
        neg_dist   = distances[mask]
        active_pct = (torch.relu(margin + pos_distances - distances) * mask).gt(0) \
                         .float().sum() / (B * (B - 1)) * 100
        dist_stats = {
            'pos_mean':   pos_distances.mean().item(),
            'neg_mean':   neg_dist.mean().item(),
            'gap':        (neg_dist.mean() - pos_distances.mean()).item(),
            'active_pct': active_pct.item(),
        }

    return loss, dist_stats


def margin_inbatch_loss(query_embs, doc_embs, mean, std,
                         covariance_matrix, margin, batch_idx, cfg):
    B        = query_embs.shape[0]
    pair_dim = mean.shape[1]

    identity = torch.eye(pair_dim, device=covariance_matrix.device) * cfg.regularization_term
    reg_cov  = covariance_matrix + identity

    if cfg.use_diagonal_covariance:
        inv_diag = 1.0 / torch.diag(reg_cov)
    else:
        inv_cov = torch.inverse(reg_cov)

    query_exp = query_embs.unsqueeze(1).expand(-1, B, -1)
    doc_exp   = doc_embs.unsqueeze(0).expand(B, -1, -1)

    if cfg.operation == 'concat':
        pairs = torch.cat([query_exp, doc_exp], dim=2)
    elif cfg.operation == 'diff':
        pairs = query_exp - doc_exp
    elif cfg.operation == 'sum':
        pairs = query_exp + doc_exp

    norm_flat = ((pairs - mean) / (std + 1e-7)).view(-1, pair_dim)

    if cfg.use_diagonal_covariance:
        mah_sq = torch.sum((norm_flat ** 2) * inv_diag, dim=1)
    else:
        mah_sq = torch.sum(norm_flat @ inv_cov * norm_flat, dim=1)

    distances     = mah_sq.view(B, B)
    pos_distances = torch.diag(distances).unsqueeze(1)
    mask          = ~torch.eye(B, dtype=torch.bool, device=distances.device)

    violations = torch.relu(margin + pos_distances - distances)
    loss       = (violations * mask).sum() / (B * (B - 1))

    with torch.no_grad():
        neg_dist   = distances[mask]
        active_pct = (violations[mask] > 0).float().mean() * 100
        dist_stats = {
            'pos_mean':   pos_distances.mean().item(),
            'neg_mean':   neg_dist.mean().item(),
            'gap':        (neg_dist.mean() - pos_distances.mean()).item(),
            'active_pct': active_pct.item(),
        }

    return loss, dist_stats


def infonce_mahalanobis_loss(query_embs, doc_embs, mean, std,
                              covariance_matrix, temperature, batch_idx, cfg):
    B        = query_embs.shape[0]
    pair_dim = mean.shape[1]

    identity = torch.eye(pair_dim, device=covariance_matrix.device) * cfg.regularization_term
    reg_cov  = covariance_matrix + identity
    inv_cov  = torch.inverse(reg_cov)

    query_exp = query_embs.unsqueeze(1).expand(-1, B, -1)
    doc_exp   = doc_embs.unsqueeze(0).expand(B, -1, -1)

    if cfg.operation == 'concat':
        pairs = torch.cat([query_exp, doc_exp], dim=2)
    elif cfg.operation == 'diff':
        pairs = query_exp - doc_exp
    elif cfg.operation == 'sum':
        pairs = query_exp + doc_exp

    norm_flat = ((pairs - mean) / (std + 1e-7)).view(-1, pair_dim)
    mah_sq    = torch.sum(norm_flat @ inv_cov * norm_flat, dim=1)
    distances = mah_sq.view(B, B)

    logits = -distances / temperature
    labels = torch.arange(B, device=logits.device)
    loss   = F.cross_entropy(logits, labels)

    with torch.no_grad():
        pos_distances = torch.diag(distances)
        mask          = ~torch.eye(B, dtype=torch.bool, device=distances.device)
        neg_dist      = distances[mask]
        # active_pct = in-batch top-1 accuracy (pos ranked #1 among all docs)
        in_batch_acc  = (torch.argmin(distances, dim=1) == labels).float().mean() * 100
        dist_stats = {
            'pos_mean':   pos_distances.mean().item(),
            'neg_mean':   neg_dist.mean().item(),
            'gap':        (neg_dist.mean() - pos_distances.mean()).item(),
            'active_pct': in_batch_acc.item(),   # repurposed: in-batch acc for infonce
        }

    return loss, dist_stats


LOSS_REGISTRY = {
    'soft_margin': soft_margin_inbatch_loss,
    'hard_margin': margin_inbatch_loss,
    'infonce':     infonce_mahalanobis_loss,
}


## Cell 8: Window Statistics

In [ ]:
# ============================================================
# CELL 8: Window Statistics
# ============================================================

def update_window(window_data, new_pairs, window_size):
    '''FIFO buffer of positive pair embeddings for covariance estimation.'''
    window_data.append(new_pairs.detach())
    if len(window_data) > window_size:
        window_data.pop(0)
    return window_data


def compute_window_statistics(window_data, use_diagonal=False):
    '''
    Compute mean, full/diagonal covariance, and std from the window buffer.

    Full covariance captures Q x D cross-correlations — the core signal
    for the concat operation. Diagonal covariance loses this but is faster.

    Returns: mean [1, dim], covariance_matrix [dim, dim], std [1, dim]
    '''
    concatenated = torch.cat(window_data, dim=0)  # [N, dim]
    mean = torch.mean(concatenated, dim=0, keepdim=True)  # [1, dim]
    std  = torch.std(concatenated,  dim=0, unbiased=False, keepdim=True)  # [1, dim]
    normalized   = (concatenated - mean) / (std + 1e-7)  # [N, dim]

    if use_diagonal:
        variance          = torch.var(normalized, dim=0, unbiased=False)  # [dim]
        covariance_matrix = torch.diag(variance)  # [dim, dim]
    else:
        covariance_matrix = torch.cov(normalized.T)  # [dim, dim]

    return mean, covariance_matrix, std

## Cell 9: Evaluation

In [ ]:
# ============================================================
# CELL 9: Evaluation
# ============================================================

def eval_ir(model, dev_dataloader, df_collection,
            mean, covariance_matrix, std, cfg, train_data=None):
    '''
    Evaluate IR using Mahalanobis distance.

    Always returns: {'MRR @ 10': float, 'Recall @ 50': float}
    When train_data is passed, also returns:
        'train_MRR @ 10', 'train_Recall @ 50'
    and logs the gap (train - dev) to detect overfitting.

    Corpus is encoded ONCE and reused for both dev and train evaluation.
    '''
    model.eval()
    device = next(model.parameters()).device

    # Collect dev queries
    all_queries   = []
    all_cord_uids = []
    for batch in dev_dataloader:
        queries, cord_uids = batch
        all_queries.extend(queries)
        all_cord_uids.extend(cord_uids)

    corpus_texts     = df_collection['input_text'].tolist()
    corpus_cord_uids = df_collection['cord_uid'].values
    logger.info(f'Evaluating {len(all_queries)} dev queries vs {len(corpus_texts)} docs...')

    # Encode corpus ONCE — reused for both dev and train eval
    with torch.no_grad():
        logger.info('Encoding corpus...')
        corpus_embeddings = cfg.encode_fn(
            corpus_texts, model, device, cfg,
            batch_size=cfg.eval_batch_size_corpus
        )

    mean              = mean.to(device)
    std               = std.to(device)
    covariance_matrix = covariance_matrix.to(device)

    base_dim     = corpus_embeddings.shape[1]
    expected_dim = base_dim * 2 if cfg.operation == 'concat' else base_dim

    if mean.shape[1] != expected_dim:
        raise ValueError(f'Stats dim mismatch: expected {expected_dim}, got mean={mean.shape[1]}')

    reg_cov = covariance_matrix + torch.eye(expected_dim, device=device) * cfg.regularization_term

    if cfg.use_diagonal_covariance:
        inv_diag = 1.0 / torch.diag(reg_cov)
    else:
        inv_cov = torch.inverse(reg_cov)

    def compute_rankings(query_texts, query_cord_uids, split_name='dev'):
        '''Rank queries against corpus using Mahalanobis distance. Returns (mrr@10, recall@50).'''
        with torch.no_grad():
            logger.info(f'Encoding {split_name} queries ({len(query_texts)})...')
            query_embeddings = cfg.encode_fn(
                query_texts, model, device, cfg,
                batch_size=cfg.eval_batch_size_query
            )

        mrr_list    = []
        recall_list = []

        with torch.no_grad():
            for q_start in range(0, len(query_cord_uids), cfg.eval_batch_size_query):
                q_end      = min(q_start + cfg.eval_batch_size_query, len(query_cord_uids))
                batch_q    = query_embeddings[q_start:q_end]
                batch_uids = query_cord_uids[q_start:q_end]

                q_exp = batch_q.unsqueeze(1).expand(-1, corpus_embeddings.size(0), -1)
                d_exp = corpus_embeddings.unsqueeze(0).expand(batch_q.size(0), -1, -1)

                if cfg.operation == 'concat':
                    pairs = torch.cat([q_exp, d_exp], dim=2)
                elif cfg.operation == 'diff':
                    pairs = q_exp - d_exp
                elif cfg.operation == 'sum':
                    pairs = q_exp + d_exp

                norm_flat = ((pairs - mean) / (std + 1e-7)).view(-1, expected_dim)

                if cfg.use_diagonal_covariance:
                    mah_sq = torch.sum((norm_flat ** 2) * inv_diag, dim=1)
                else:
                    mah_sq = torch.sum(norm_flat @ inv_cov * norm_flat, dim=1)

                mah_sq = mah_sq.view(batch_q.size(0), -1)

                for i, correct_uid in enumerate(batch_uids):
                    ranked_indices = torch.argsort(mah_sq[i]).cpu().tolist()
                    ranked_uids    = [corpus_cord_uids[idx] for idx in ranked_indices]
                    try:
                        rank = ranked_uids.index(correct_uid) + 1
                        mrr_list.append(1.0 / rank if rank <= 10 else 0.0)
                        recall_list.append(1 if rank <= 50 else 0)
                    except ValueError:
                        mrr_list.append(0.0)
                        recall_list.append(0)

        return np.mean(mrr_list), np.mean(recall_list)

    # --- Dev evaluation ---
    logger.info(f'--- Dev Evaluation ({len(all_queries)} queries) ---')
    dev_mrr, dev_recall = compute_rankings(all_queries, all_cord_uids, 'dev')
    logger.info(f'  Dev  MRR@10={dev_mrr:.4f}  Recall@50={dev_recall:.4f}')

    results = {
        'MRR @ 10':    dev_mrr,
        'Recall @ 50': dev_recall,
    }

    # --- Train evaluation (overfitting detection) ---
    if train_data is not None:
        # Extract unique (query, cord_uid) pairs — train_data[i] = (query, positive, cord_uid)
        seen = set()
        unique_queries, unique_uids = [], []
        for q, _, uid in train_data:
            key = (q, uid)
            if key not in seen:
                seen.add(key)
                unique_queries.append(q)
                unique_uids.append(uid)

        # Optionally cap the number of train queries evaluated
        if cfg.train_eval_samples and len(unique_queries) > cfg.train_eval_samples:
            indices       = random.sample(range(len(unique_queries)), cfg.train_eval_samples)
            unique_queries = [unique_queries[i] for i in indices]
            unique_uids    = [unique_uids[i]    for i in indices]

        logger.info(f'--- Train Evaluation ({len(unique_queries)} queries) ---')
        train_mrr, train_recall = compute_rankings(unique_queries, unique_uids, 'train')
        gap = train_recall - dev_recall
        logger.info(f'  Train MRR@10={train_mrr:.4f}  Recall@50={train_recall:.4f}')
        logger.info(f'  >>> Gap (train - dev) Recall@50: {gap:+.4f} <<<')

        results['train_MRR @ 10']    = train_mrr
        results['train_Recall @ 50'] = train_recall

    return results

## Cell 10: Save / Load

In [ ]:
# ============================================================
# CELL 10: Save / Load
# ============================================================

def save_model_and_stats(model, mean, covariance_matrix, std,
                          operation, model_path, cfg,
                          run_id=None, window_data=None):
    '''Save model weights + statistics to paired files (*model_path* and *model_path_stats.pt*).'''
    os.makedirs(os.path.dirname(model_path), exist_ok=True)
    torch.save(model.state_dict(), model_path)

    base, ext  = os.path.splitext(model_path)
    stats_path = f'{base}_stats{ext}'

    save_dict = {
        'mean':                    mean.cpu(),
        'covariance_matrix':       covariance_matrix.cpu(),
        'std':                     std.cpu(),
        'operation':               operation,
        'experiment_name':         cfg.experiment_name,
        'run_number':              cfg.run_number,        # add
        'dataset_name':            cfg.dataset_name,      # add
        'model_id':                cfg.model_id,
        'loss_fn':                 cfg.loss_fn,
        'margin':                  cfg.margin,
        'use_diagonal_covariance': cfg.use_diagonal_covariance,
        'train_path':              cfg.train_path,
        'wandb_run_id':            run_id,
        'timestamp':               datetime.now().isoformat(),
    }


    if window_data is not None:
        save_dict['window_data'] = [w.cpu() for w in window_data]
        logger.info(f'Window data saved: {len(window_data)} batches')

    torch.save(save_dict, stats_path)
    logger.info(f'Saved model -> {model_path}')
    logger.info(f'Saved stats -> {stats_path}')
    return stats_path


def load_model_and_stats(model, model_path, device):
    '''Load model weights + statistics. Returns (mean, cov, std, operation, window_data).'''
    model.load_state_dict(torch.load(model_path, map_location=device))

    base, ext  = os.path.splitext(model_path)
    stats_path = f'{base}_stats{ext}'
    stats      = torch.load(stats_path, map_location=device)

    mean              = stats['mean'].to(device)
    covariance_matrix = stats['covariance_matrix'].to(device)
    std               = stats['std'].to(device)
    operation         = stats.get('operation', 'concat')
    window_data       = ([w.to(device) for w in stats['window_data']]
                         if 'window_data' in stats else None)

    logger.info(f'Loaded <- {model_path}')
    logger.info(f'  operation={operation}, '
                f'cov={"diagonal" if stats.get("use_diagonal_covariance") else "full"}')
    if 'experiment_name' in stats:
        logger.info(f'  experiment={stats["experiment_name"]}, '
                    f'loss_fn={stats.get("loss_fn", "unknown")}')

    return mean, covariance_matrix, std, operation, window_data

## Cell 11: Train

In [ ]:
# ============================================================
# CELL 11: Training
# ============================================================

def train(model, train_dataloader, dev_dataloader, df_collection,
          optimizer, scheduler, cfg, window_data=None, train_data=None,
          best=-float('inf'), eval_stall=0):

    '''
    Train for one epoch.
    Returns: (early_stopped: bool, window_data, mean, covariance_matrix, std)
    No global variables — all state is local and returned.

    train_data: if provided, eval_ir will also evaluate on training queries
                each epoch to detect overfitting (gap = train_recall - dev_recall).
    '''
    if window_data is None:
        window_data = []

    global_step   = 0
    total_batches = len(train_dataloader)
    mean = covariance_matrix = std = None
    device        = next(model.parameters()).device
    eval_interval = total_batches  # evaluate once per epoch

    loss_fn    = LOSS_REGISTRY[cfg.loss_fn]
    loss_param = cfg.margin if cfg.loss_fn != 'infonce' else cfg.temperature

    model.train()
    optimizer.zero_grad()

    for batch_idx, batch in enumerate(tqdm(train_dataloader, desc='Training'), start=1):
        current_lr = optimizer.param_groups[0]['lr']
        torch.cuda.empty_cache()

        query, positive = batch[0], batch[1]

        # Model-agnostic forward — API is hidden in cfg.forward_fn
        query_emb    = cfg.forward_fn(query,    model, device, cfg)  # [B, emb_dim]
        if model.training and cfg.query_dropout > 0:
            query_emb = F.dropout(query_emb, p=cfg.query_dropout, training=True)

        positive_emb = cfg.forward_fn(positive, model, device, cfg)  # [B, emb_dim]

        # Build positive pairs for window update
        positive_pairs = apply_query_doc_operation(query_emb, positive_emb, cfg.operation)
        out_detached   = positive_pairs.detach()

        window_data = update_window(window_data, out_detached, cfg.window_size)
        mean, covariance_matrix, std = compute_window_statistics(
            window_data, use_diagonal=cfg.use_diagonal_covariance
        )

        loss, dist_stats = loss_fn(query_emb, positive_emb, mean, std,
                                    covariance_matrix, loss_param, batch_idx, cfg)

        global_step += 1

        # Gradient accumulation
        (loss / cfg.grad_accum_steps).backward()

        if (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == total_batches):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.max_grad_norm)
            optimizer.step()
            optimizer.zero_grad()
            try:
                scheduler.step()
            except ValueError:
                pass  # scheduler exhausted on final batch

        if batch_idx % 10 == 0:
            logger.info(f'[Batch {batch_idx}/{total_batches}] step={global_step}  '
                        f'loss={loss.item():.4f}  lr={current_lr:.2e}  '
                        f'pos={dist_stats["pos_mean"]:.1f}  neg={dist_stats["neg_mean"]:.1f}  '
                        f'gap={dist_stats["gap"]:.1f}  active={dist_stats["active_pct"]:.1f}%')

        if wandb.run:
            wandb.log({'train/loss':            loss.item(),
                      'train/lr':             current_lr,
                      'distances/pos_mean':   dist_stats['pos_mean'],
                      'distances/neg_mean':   dist_stats['neg_mean'],
                      'distances/gap':        dist_stats['gap'],
                      'distances/active_pct': dist_stats['active_pct'],
                      'step': global_step})

        # Evaluation (once per epoch)
        if (batch_idx % eval_interval == 0) or (batch_idx == total_batches):
            model.eval()
            metrics = eval_ir(model, dev_dataloader, df_collection,
                              mean, covariance_matrix, std, cfg,
                              train_data=train_data)   # pass train_data for overfitting detection
            recall_at_50 = metrics['Recall @ 50']
            mrr_at_10    = metrics['MRR @ 10']
            logger.info(f'[Eval] step={global_step}  '
                        f'Dev MRR@10={mrr_at_10:.4f}  Recall@50={recall_at_50:.4f}')

            log_dict = {
                'mrr/dev':    mrr_at_10,
                'recall/dev': recall_at_50,
                'step': global_step,
            }


            # Log train metrics and gap when available
            if 'train_Recall @ 50' in metrics:
                train_mrr    = metrics['train_MRR @ 10']
                train_recall = metrics['train_Recall @ 50']
                gap          = train_recall - recall_at_50
                logger.info(f'         Train MRR@10={train_mrr:.4f}  Recall@50={train_recall:.4f}  '
                            f'Gap={gap:+.4f}')
                log_dict['mrr/train']    = train_mrr
                log_dict['recall/train'] = train_recall
                log_dict['recall/gap']   = gap


            if wandb.run:
                wandb.log(log_dict)

            model.train()

            if recall_at_50 > best:
                best       = recall_at_50
                eval_stall = 0
                save_model_and_stats(
                    model, mean, covariance_matrix, std,
                    cfg.operation, cfg.save_path, cfg,
                    run_id=wandb.run.id if wandb.run else None,
                    window_data=window_data,
                )
                logger.info(f'  New best Recall@50: {best:.4f}')
            else:
                eval_stall += 1
                logger.info(f'  No improvement ({eval_stall}/{cfg.eval_patience})')
                if eval_stall >= cfg.eval_patience:
                    logger.info(f'Early stopping at step {global_step}')
                    return True, window_data, mean, covariance_matrix, std, best, eval_stall

    return False, window_data, mean, covariance_matrix, std, best, eval_stall

## Cell 12: Experiment Runner

In [ ]:
# ============================================================
# CELL 12: Experiment Runner
# ============================================================

def run_experiment(cfg):
    '''Full training pipeline from a single ExperimentConfig.'''
    set_seed(cfg.seed)
    os.makedirs(cfg.save_dir, exist_ok=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    logger.info(f'=== {cfg.experiment_name} ===')
    logger.info(f'Model: {cfg.model_id} | Loss: {cfg.loss_fn} | '
                f'Op: {cfg.operation} | Cov: {"diagonal" if cfg.use_diagonal_covariance else "full"}')

    # Data
    train_data    = load_data(True,  cfg.train_path, use_triplets=False)
    dev_data      = load_data(False, cfg.dev_path)
    df_collection = pd.read_csv(cfg.collection_path)
    df_collection['input_text'] = (
        'title: '   + df_collection['title_processed'] +
        ' | text: ' + df_collection['abstract_processed']
    )
    logger.info(f'Train: {len(train_data):,} pairs | Dev: {len(dev_data):,} queries | '
                f'Corpus: {len(df_collection):,} docs')
    if cfg.train_eval_samples:
        logger.info(f'Train eval: {cfg.train_eval_samples} sampled queries per epoch')
    else:
        logger.info(f'Train eval: all {len(train_data):,} train queries per epoch')

    # Model (via cfg.build_fn — model-type agnostic)
    model       = cfg.build_fn(cfg, device)
    window_data = []
    if cfg.resume_model_path:
        mean, cov, std, _, window_data = load_model_and_stats(
            model, cfg.resume_model_path, device
        )
        window_data = window_data or []
        logger.info(f'Resumed: {len(window_data)} window batches')

    # DataLoaders
    if cfg.hard_neg_path and os.path.exists(cfg.hard_neg_path):
        hard_neg_data = torch.load(cfg.hard_neg_path)
        hard_negs     = hard_neg_data['hard_negs']
        logger.info(f'Loaded hard negatives from {cfg.hard_neg_path}')
        train_sampler = HardNegBatchSampler(
            train_data, hard_negs, batch_size=cfg.batch_size,
            num_hard_per_query=cfg.num_hard_per_query,
        )
    else:
        train_sampler = GreedyUniqueBatchSampler(train_data, batch_size=cfg.batch_size)

    train_dataloader = DataLoader(train_data, batch_sampler=train_sampler)
    dev_dataloader   = DataLoader(dev_data,   batch_size=cfg.batch_size)
    logger.info(f'Batches per epoch: {len(train_dataloader)}')

    # Optimizer + Scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    steps_per_epoch = (
        (len(train_dataloader) + cfg.grad_accum_steps - 1) // cfg.grad_accum_steps
    )

    scheduler = OneCycleLR(
        optimizer,
        max_lr              = cfg.scheduler_max_lr,
        steps_per_epoch     = steps_per_epoch,
        epochs              = cfg.total_epochs,
        pct_start           = cfg.scheduler_pct_start,
        anneal_strategy     = cfg.scheduler_anneal_strategy,
        div_factor          = cfg.scheduler_div_factor,
        final_div_factor    = cfg.scheduler_final_div_factor,
    )
    if cfg.resume_model_path and cfg.start_epoch > 0:
        for _ in range(cfg.start_epoch * steps_per_epoch):
            scheduler.step()
        logger.info(f'Scheduler fast-forwarded to epoch {cfg.start_epoch}')

    # W&B (filter out un-serializable Callable fields)
    if cfg.wandb_project:
        wandb_config = {k: v for k, v in vars(cfg).items() if not callable(v)}
        wandb.init(
            project = cfg.wandb_project,
            group   = cfg.wandb_group,
            config  = wandb_config,
            name    = cfg.experiment_name,
            tags    = [
                cfg.dataset_name,
                cfg.loss_fn,
                cfg.model_id.split('/')[-1],
                f'run_{cfg.run_number:03d}',
            ],
            reinit  = True,
        )

        logger.info(f'W&B run: {wandb.run.id}')

    # Training loop
    best = -float('inf')
    eval_stall = 0
    for epoch in range(cfg.start_epoch, cfg.total_epochs):
        logger.info(f'--- Epoch {epoch+1}/{cfg.total_epochs} ---')
        early_stopped, window_data, _, _, _, best, eval_stall = train(
            model, train_dataloader, dev_dataloader, df_collection,
            optimizer, scheduler, cfg, window_data=window_data,
            train_data=train_data, best=best, eval_stall=eval_stall,
        )
        if early_stopped:
            logger.info(f'Stopped early at epoch {epoch+1}')
            break

    logger.info(f'Training complete. Best model at {cfg.save_path}')
    if wandb.run:
        wandb.finish()

## Cell 13: Experiment Definitions

> Edit configs here. Change the name in Cell 15 to switch experiments.

###MiniLM

In [ ]:
# from transformers import AutoTokenizer

# tok_minilm = AutoTokenizer.from_pretrained('microsoft/MiniLM-L12-H384-uncased')

# CFG_MINILM = ExperimentConfig(
#     run_number         = 6,
#     dataset_name       = 'clef2025',
#     wandb_group        = 'ClaD_MiniLM',
#     wandb_project      = 'ClaD_Finale',
#     model_id           = 'microsoft/MiniLM-L12-H384-uncased',
#     build_fn           = hf_build,
#     forward_fn         = hf_forward,
#     encode_fn          = hf_encode,
#     tokenizer          = tok_minilm,
#     max_seq_length     = 512,

#     # Training
#     batch_size         = 128,
#     total_epochs       = 12,
#     seed               = 42,
#     max_grad_norm      = 1.0,

#     # Optimizer
#     lr                         = 0,
#     scheduler_max_lr           = 6e-5,
#     scheduler_pct_start        = 0.3,
#     scheduler_div_factor       = 10.0,
#     scheduler_final_div_factor = 2.0,
#     scheduler_anneal_strategy  = 'cos',

#     # Loss
#     loss_fn            = 'infonce',
#     temperature        = 80.0,

#     # Window
#     window_size        = 60,

#     # Regularization
#     query_dropout      = 0.05,
#     projection_dropout = 0.1,
#     weight_decay       = 0.02,

#     # Eval
#     eval_patience      = 3,
# )

# # from dataclasses import replace

# # CFG_MINILM_INFONCE = replace(CFG_MINILM,
# #     run_number      = 1,
# #     loss_fn         = 'infonce',
# #     temperature     = 40.0,
# # )

### BERT

In [ ]:
from transformers import AutoTokenizer

# tok_bert = AutoTokenizer.from_pretrained('bert-base-uncased')

# CFG_BERT_HARD = ExperimentConfig(
#     run_number         = 5,
#     dataset_name       = 'clef2025',
#     wandb_group        = 'ClaD_BERT',
#     wandb_project      = 'ClaD_Finale',
#     model_id           = 'bert-base-uncased',
#     build_fn           = hf_build,
#     forward_fn         = hf_forward,
#     encode_fn          = hf_encode,
#     tokenizer          = tok_bert,
#     max_seq_length     = 512,

#     batch_size         = 64,
#     total_epochs       = 10,
#     lr                 = 1e-5,
#     scheduler_max_lr   = 3e-5,
#     scheduler_div_factor= 10.0,
#     scheduler_pct_start        = 0.1,
#     scheduler_final_div_factor = 2.0,

#     loss_fn            = 'infonce',
#     temperature        = 30.0,
#     window_size        = 60,

#     query_dropout      = 0.0,
#     projection_dropout = 0.1,
#     weight_decay       = 0.05,
#     eval_patience      = 3,

#     resume_model_path = '/content/drive/MyDrive/Parth/CLEF_prvt/Spring 26_finale/Models/ClaD/clef2025/exp_002_bert-base-un_infonce_clef2025/best_model.pt',
#     hard_neg_path     = '/content/drive/MyDrive/Parth/CLEF_prvt/Spring 26_finale/Models/ClaD/clef2025/exp_002_bert-base-un_infonce_clef2025/hard_negatives.pt',
#     num_hard_per_query= 16,
# )


tok_bert = AutoTokenizer.from_pretrained('allenai/scibert_scivocab_uncased')

CFG_SCIBERT = ExperimentConfig(
    run_number         = 2,
    dataset_name       = 'clef2025',
    wandb_group        = 'ClaD_SciBERT',
    wandb_project      = 'ClaD_Finale',
    model_id           = 'allenai/scibert_scivocab_uncased',
    build_fn           = hf_build,
    forward_fn         = hf_forward,
    encode_fn          = hf_encode,
    tokenizer          = tok_bert,
    max_seq_length     = 512,

    batch_size         = 128,
    total_epochs       = 5,
    lr                 = 1e-5,
    scheduler_max_lr   = 5e-5,
    scheduler_div_factor= 10.0,
    scheduler_pct_start        = 0.3,
    scheduler_final_div_factor = 2.0,

    loss_fn            = 'infonce',
    temperature        = 40.0,
    window_size        = 60,

    query_dropout      = 0.0,
    projection_dropout = 0.15,
    weight_decay       = 0.1,
    eval_patience      = 3,

    resume_model_path = None,
    hard_neg_path     = None,
    num_hard_per_query= None,
)



###Other models

In [ ]:
# from transformers import AutoTokenizer

# ============================================================
# Tokenizers (instantiate once, reuse across configs)
# ============================================================

# tok_bert   = AutoTokenizer.from_pretrained('bert-base-uncased')

# tok_mpnet  = AutoTokenizer.from_pretrained('microsoft/mpnet-base')

# ============================================================
# Base configs — one per backbone, soft_margin default
# ============================================================

# CFG_BERT = ExperimentConfig(
#     experiment_name  = 'bert_soft_margin',
#     wandb_group      = 'ClaD_BERT',
#     wandb_project    = 'ClaD_Finale',
#     model_id         = 'bert-base-uncased',
#     build_fn         = hf_build,
#     forward_fn       = hf_forward,
#     encode_fn        = hf_encode,
#     tokenizer        = tok_bert,
#     max_seq_length   = 512,
#     batch_size       = 32,
#     lr               = 2e-5,
#     scheduler_max_lr = 5e-5,
#     loss_fn          = 'soft_margin',
#     margin           = 150.0,
#     total_epochs     = 10,
# )

# CFG_MPNET = ExperimentConfig(
#     experiment_name  = 'mpnet_soft_margin',
#     wandb_group      = 'ClaD_MPNET',
#     wandb_project    = 'ClaD_Finale',
#     model_id         = 'microsoft/mpnet-base',
#     build_fn         = hf_build,
#     forward_fn       = hf_forward,
#     encode_fn        = hf_encode,
#     tokenizer        = tok_mpnet,
#     max_seq_length   = 512,
#     batch_size       = 32,
#     lr               = 2e-5,
#     scheduler_max_lr = 5e-5,
#     loss_fn          = 'soft_margin',
#     margin           = 150.0,
#     total_epochs     = 10,
# )

# ============================================================
# Ablations — copy base, override loss + name
# ============================================================

# from dataclasses import replace

# CFG_BERT_HARD     = replace(CFG_BERT,   experiment_name='bert_hard_margin',   loss_fn='hard_margin')
# CFG_MINILM_HARD   = replace(CFG_MINILM, experiment_name='minilm_hard_margin', loss_fn='hard_margin')
# CFG_MPNET_HARD    = replace(CFG_MPNET,  experiment_name='mpnet_hard_margin',  loss_fn='hard_margin')

# CFG_BERT_INFONCE   = replace(CFG_BERT,   experiment_name='bert_infonce',   loss_fn='infonce', temperature=50.0)
# CFG_MPNET_INFONCE  = replace(CFG_MPNET,  experiment_name='mpnet_infonce',  loss_fn='infonce', temperature=50.0)


# --- Example: resume from a previous experiment ---
# CFG_MINILM_RESUME = ExperimentConfig(
#     experiment_name    = 'exp_005_minilm_resume',
#     model_id           = 'sentence-transformers/all-MiniLM-L6-v2',
#     resume_model_path  = f'{MODEL_DIR}/exp_001_minilm_soft_margin/best_model.pt',
#     start_epoch        = 5,
#     total_epochs       = 10,
#     loss_fn            = 'soft_margin',
#     margin             = 150.0,
# )

## Cell 14: Smoke Test

> **Run `verify_pipeline(CFG)` before every `run_experiment(CFG)` call.**
> Takes ~30 seconds. Catches all config/shape/data errors before a multi-hour training run.

In [ ]:
# ============================================================
# CELL 14: Smoke Test
# verify_pipeline(cfg) — 9 checks, ~30 seconds
# Run after ANY config change: model / dataset / loss / operation
# ============================================================

def verify_pipeline(cfg):
    '''9-step component test. Catches mismatches before a full training run.'''
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    set_seed(cfg.seed)
    passed = 0
    model  = None

    try:
        # ---- 1. Model builds without error ----
        print('1. Building model...')
        model    = cfg.build_fn(cfg, device)
        n_params = sum(p.numel() for p in model.parameters())
        print(f'   OK: {cfg.model_id}, params={n_params:,}')
        passed += 1

        # ---- 2. Forward pass: correct shape, no NaN ----
        print('2. Forward pass...')
        model.train()
        dummy        = ['This is a test query sentence', 'Another test sentence here']
        emb          = cfg.forward_fn(dummy, model, device, cfg)
        expected_dim = 128 if cfg.operation == 'concat' else 256
        assert emb.shape == (2, expected_dim), \
            f'Expected (2, {expected_dim}), got {emb.shape}'
        assert not torch.isnan(emb).any(), 'NaN in embeddings'
        print(f'   OK: shape={emb.shape}, norms={[round(n,3) for n in emb.norm(dim=1).tolist()]}')
        passed += 1

        # ---- 3. Operation: correct pair_dim ----
        print('3. Query-doc operation...')
        pair              = apply_query_doc_operation(emb[:1], emb[1:], cfg.operation)
        expected_pair_dim = expected_dim * 2 if cfg.operation == 'concat' else expected_dim
        assert pair.shape[1] == expected_pair_dim, \
            f'Expected pair_dim={expected_pair_dim}, got {pair.shape[1]}'
        print(f'   OK: pair_dim={pair.shape[1]}')
        passed += 1

        # ---- 4. Window statistics shapes ----
        print('4. Window statistics...')
        window = []
        for _ in range(3):
            fake   = torch.randn(cfg.batch_size, expected_pair_dim, device=device)
            window = update_window(window, fake, cfg.window_size)
        m, c, s = compute_window_statistics(window, use_diagonal=cfg.use_diagonal_covariance)
        assert m.shape == (1, expected_pair_dim), f'mean shape {m.shape}'
        assert c.shape == (expected_pair_dim, expected_pair_dim), f'cov shape {c.shape}'
        assert s.shape == (1, expected_pair_dim), f'std shape {s.shape}'
        print(f'   OK: mean={m.shape}, cov={c.shape}')
        passed += 1

        # ---- 5. Loss: scalar, positive, backward works ----
        print(f'5. Loss ({cfg.loss_fn})...')
        q_texts    = ['query one', 'query two', 'query three', 'query four']
        d_texts    = ['doc one',   'doc two',   'doc three',   'doc four']
        q_emb      = cfg.forward_fn(q_texts, model, device, cfg)
        d_emb      = cfg.forward_fn(d_texts, model, device, cfg)
        pairs      = apply_query_doc_operation(q_emb, d_emb, cfg.operation)
        m, c, s    = compute_window_statistics([pairs.detach()],
                                               use_diagonal=cfg.use_diagonal_covariance)
        loss_fn    = LOSS_REGISTRY[cfg.loss_fn]
        loss_param = cfg.margin if cfg.loss_fn != 'infonce' else cfg.temperature
        loss, dist_stats = loss_fn(q_emb, d_emb, m, s, c, loss_param, 0, cfg)
        assert loss.dim() == 0,       'Loss not scalar'
        assert not torch.isnan(loss), f"   FAIL: loss is NaN"
        assert loss.item() >= 0, f"   FAIL: loss should be non-negative, got {loss.item()}"
        loss.backward()
        print(f'   OK: loss={loss.item():.4f}  pos={dist_stats["pos_mean"]:.1f}  '
              f'neg={dist_stats["neg_mean"]:.1f}  active={dist_stats["active_pct"]:.1f}%  backward OK')

        # ---- 6. Encode function (eval mode) ----
        print('6. Encode function...')
        model.eval()
        with torch.no_():
            enc = cfg.encode_fn(q_texts, model, device, cfg, batch_size=2)
        assert enc.shape == (4, expected_dim), \
            f'Expected (4, {expected_dim}), got {enc.shape}'
        print(f'   OK: {enc.shape}')
        passed += 1

        # ---- 7. Data loading ----
        print('7. Data loading...')
        train_data = load_data(True,  cfg.train_path, use_triplets=False)
        dev_data   = load_data(False, cfg.dev_path)
        assert len(train_data) > 0,     'Empty train data'
        assert len(dev_data)   > 0,     'Empty dev data'
        assert len(train_data[0]) == 3, f'Train tuple len={len(train_data[0])}, expected 3'
        assert len(dev_data[0])   == 2, f'Dev tuple len={len(dev_data[0])}, expected 2'
        print(f'   OK: train={len(train_data):,} pairs, dev={len(dev_data):,} queries')
        passed += 1

        # ---- 8. Save/Load round-trip ----
        print('8. Save/Load round-trip...')
        with tempfile.TemporaryDirectory() as tmpdir:
            tmp_path = os.path.join(tmpdir, 'smoke_test.pt')
            save_model_and_stats(model, m, c, s, cfg.operation, tmp_path,
                                  cfg=cfg, window_data=[pairs.detach()])
            m2, c2, s2, op2, _ = load_model_and_stats(model, tmp_path, device)
            assert op2 == cfg.operation, f'Op mismatch: {op2} vs {cfg.operation}'
            assert torch.allclose(m, m2, atol=1e-5), 'Mean mismatch after load'
        print('   OK: round-trip matches')
        passed += 1

        # ---- 9. Sampler: unique UIDs per batch ----
        print('9. GreedyUniqueBatchSampler...')
        sampler     = GreedyUniqueBatchSampler(train_data, batch_size=cfg.batch_size)
        first_batch = next(iter(sampler))
        uids        = [str(train_data[i][2]) for i in first_batch]
        assert len(uids) == len(set(uids)), f'Duplicate UIDs in batch'
        print(f'   OK: batch_size={len(first_batch)}, all UIDs unique')
        passed += 1

    except AssertionError as e:
        print(f'\n   FAIL: {e}')
    except Exception as e:
        print(f'\n   ERROR: {type(e).__name__}: {e}')
        raise
    finally:
        if model is not None:
            del model
        torch.cuda.empty_cache()

    sep = '=' * 52
    if passed == 9:
        print(f'\n{sep}')
        print(f'ALL 9 CHECKS PASSED')
        print(f'  experiment : {cfg.experiment_name}')
        print(f'  model      : {cfg.model_id}')
        print(f'  loss       : {cfg.loss_fn}  margin={cfg.margin}')
        print(f'  operation  : {cfg.operation}')
        print(f'{sep}')
    else:
        print(f'\n{passed}/9 checks passed')

## Cell 15: Run

> Change the config name here only. Run Cell 14 first.

In [ ]:
# ============================================================
# CELL 15: Run
# Change the config name — nothing else
# ============================================================

# verify_pipeline(CFG_BERT)
run_experiment(CFG_SCIBERT)

## Cell 16: Hard Negative Mining

Run after training to mine hard negatives for the next experiment.

In [ ]:
# ============================================================
# CELL 16: Hard Negative Mining
# Run after a completed experiment to mine hard negatives
# ============================================================

def mine_hard_negatives(model, train_data, df_collection, mean, covariance_matrix, std,
                         cfg, hard_neg_start=50, hard_neg_end=200, save_path=None):
    '''
    Mine hard negatives using the current model's Mahalanobis rankings.
    Returns docs ranked [hard_neg_start, hard_neg_end) for each query,
    excluding the query's own positive document.
    '''
    model.eval()
    device = next(model.parameters()).device

    all_queries       = [item[0] for item in train_data]
    all_positive_uids = [str(item[2]) for item in train_data]
    df_collection['input_text'] = (
        'title: '   + df_collection['title_processed'] +
        ' | text: ' + df_collection['abstract_processed']
    )
    corpus_texts      = df_collection['input_text'].tolist()
    corpus_cord_uids  = df_collection['cord_uid'].values
    uid_to_idx        = {str(uid): idx for idx, uid in enumerate(corpus_cord_uids)}

    logger.info(f'Mining hard negatives: {len(all_queries)} queries, '
                f'rank range [{hard_neg_start}, {hard_neg_end})')

    with torch.no_grad():
        query_embeddings  = cfg.encode_fn(all_queries,  model, device, cfg, batch_size=128)
        corpus_embeddings = cfg.encode_fn(corpus_texts, model, device, cfg, batch_size=64)

    mean              = mean.to(device)
    std               = std.to(device)
    covariance_matrix = covariance_matrix.to(device)

    base_dim     = query_embeddings.shape[1]
    expected_dim = base_dim * 2 if cfg.operation == 'concat' else base_dim

    reg_cov = covariance_matrix + torch.eye(expected_dim, device=device) * cfg.regularization_term
    if cfg.use_diagonal_covariance:
        inv_diag = 1.0 / torch.diag(reg_cov)
    else:
        inv_cov = torch.inverse(reg_cov)

    hard_negs      = {}
    positive_ranks = []

    with torch.no_grad():
        for q_start in tqdm(range(0, len(all_queries), 128), desc='Mining'):
            q_end   = min(q_start + 128, len(all_queries))
            batch_q = query_embeddings[q_start:q_end]

            q_exp = batch_q.unsqueeze(1).expand(-1, corpus_embeddings.size(0), -1)
            d_exp = corpus_embeddings.unsqueeze(0).expand(batch_q.size(0), -1, -1)

            if cfg.operation == 'concat':
                pairs = torch.cat([q_exp, d_exp], dim=2)
            elif cfg.operation == 'diff':
                pairs = q_exp - d_exp
            elif cfg.operation == 'sum':
                pairs = q_exp + d_exp

            norm_flat = ((pairs - mean) / (std + 1e-7)).view(-1, expected_dim)
            if cfg.use_diagonal_covariance:
                mah_sq = torch.sum((norm_flat ** 2) * inv_diag, dim=1)
            else:
                mah_sq = torch.sum(norm_flat @ inv_cov * norm_flat, dim=1)
            mah_sq = mah_sq.view(batch_q.size(0), -1)

            for i in range(batch_q.size(0)):
                dataset_idx  = q_start + i
                positive_uid = all_positive_uids[dataset_idx]
                ranked       = torch.argsort(mah_sq[i]).cpu().tolist()

                pos_idx  = uid_to_idx.get(positive_uid)
                pos_rank = ranked.index(pos_idx) + 1 if pos_idx is not None else -1
                positive_ranks.append(pos_rank)

                neg_uids = [str(corpus_cord_uids[r]) for r in ranked[hard_neg_start:hard_neg_end]
                            if str(corpus_cord_uids[r]) != positive_uid]
                hard_negs[dataset_idx] = neg_uids

    pr = np.array(positive_ranks)
    logger.info(f'Mining complete | avg negs/query={np.mean([len(v) for v in hard_negs.values()]):.1f}')
    logger.info(f'Positive ranks | median={np.median(pr):.0f}, mean={np.mean(pr):.1f}, '
                f'top-10={(pr<=10).mean():.1%}, top-50={(pr<=50).mean():.1%}')

    if save_path:
        torch.save({'hard_negs': hard_negs, 'hard_neg_start': hard_neg_start,
                    'hard_neg_end': hard_neg_end, 'timestamp': datetime.now().isoformat()},
                   save_path)
        logger.info(f'Saved -> {save_path}')

    return hard_negs

In [ ]:
CFG_BERT = ExperimentConfig(
    run_number     = 2,
    dataset_name   = 'clef2025',
    wandb_group    = 'ClaD_BERT',
    wandb_project  = 'ClaD_Finale',
    model_id       = 'bert-base-uncased',
    build_fn       = hf_build,
    forward_fn     = hf_forward,
    encode_fn      = hf_encode,
    tokenizer      = tok_bert,
    max_seq_length = 512,
    loss_fn        = 'infonce',   # needed to reconstruct experiment_name
)

model = CFG_BERT.build_fn(CFG_BERT, DEVICE)

mean, cov, std, op, window = load_model_and_stats(model, CFG_BERT.save_path, DEVICE)
train_data    = load_data(True, CFG_BERT.train_path)
df_collection = pd.read_csv(CFG_BERT.collection_path)

hard_negs = mine_hard_negatives(
    model, train_data, df_collection, mean, cov, std,
    cfg=CFG_BERT, hard_neg_start=5, hard_neg_end=30,
    save_path=f'{CFG_BERT.save_dir}/hard_negatives.pt'
)

#Test data

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CFG_BERT = ExperimentConfig(
    run_number     = 2,
    dataset_name   = 'clef2025',
    wandb_group    = 'ClaD_BERT',
    wandb_project  = 'ClaD_Finale',
    model_id       = 'bert-base-uncased',
    build_fn       = hf_build,
    forward_fn     = hf_forward,
    encode_fn      = hf_encode,
    tokenizer      = tok_bert,
    max_seq_length = 512,
    loss_fn        = 'infonce',   # needed to reconstruct experiment_name
)


# Load model
model = CFG_BERT.build_fn(CFG_BERT, device)
mean, cov, std, op, window = load_model_and_stats(model, CFG_BERT.save_path, device)

# Load test data — same format as dev (.tsv with tweet_text + cord_uid)?
test_data = load_data(False, PATH_TEST_DATA)  # ← your test path
test_dataloader = DataLoader(test_data, batch_size=128)

df_collection = pd.read_csv(CFG_BERT.collection_path)
df_collection['input_text'] = ('title: ' + df_collection['title_processed']
                                + ' | text: ' + df_collection['abstract_processed'])

# Evaluate
metrics = eval_ir(model, test_dataloader, df_collection, mean, cov, std, CFG_BERT)
print(metrics)